In [6]:
import pandas as pd
import numpy as np
import random
from faker import Faker
fake = Faker()

In [7]:
#generation of random people and their pronouns,
pronoun_groups = [
    "she/her",
    "he/him",
    "they/them",
    "xe/xer",
    "she/they",
    "he/they"
    
]
alpha = np.array([5.0, 5.0, 2.0, 0.5, 1.0, 1.0])
number_of_people = 300

# Each person gets their own Dirichlet draw = continuous pronoun affinity vector
np.random.seed(42)
affinity_matrix = np.random.dirichlet(alpha, size=number_of_people)

#additional Guassian noise
noise = np.abs(np.random.normal(0, 0.02, affinity_matrix.shape))
affinity_matrix = affinity_matrix + noise
affinity_matrix = affinity_matrix / affinity_matrix.sum(axis=1, keepdims=True)

primary_pronouns = [
    np.random.choice(pronoun_groups, p=affinity_matrix[i])
    for i in range(number_of_people)
]

people = pd.DataFrame({
    "ID":range(1, number_of_people + 1),
    "Name": [fake.name() for _ in range(number_of_people)],
    "Pronouns": primary_pronouns,
    **{f"affinity_{pg.replace('/','_')}": affinity_matrix[:, j]
       for j, pg in enumerate(pronoun_groups)}
})
people.head()

,ID,Name,Pronouns,affinity_she_her,affinity_he_him,affinity_they_them,affinity_xe_xer,affinity_she_they,affinity_he_they
0,1,Charles Gonzalez,xe/xer,0.406644,0.291097,0.099959,0.107525,0.090097,0.004678
1,2,Krista Delgado,he/they,0.334204,0.327193,0.207026,0.029623,0.055263,0.046691
2,3,Kelli Khan,he/they,0.198134,0.372973,0.314028,0.031126,0.017337,0.066402
3,4,Megan Mcconnell,she/her,0.342072,0.303781,0.117650,0.015557,0.004310,0.216629
4,5,Heather Montgomery,she/they,0.459155,0.184569,0.057818,0.067571,0.160450,0.070437


In [8]:
connective_lst = []

for i, person_id in enumerate(people["ID"]):
    number_of_connections = random.randint(1, 3)
    
    # Compute affinity similarity to all others (dot product = cosine-like weight)
    person_vec = affinity_matrix[i]
    other_ids = [pid for pid in people["ID"] if pid != person_id]
    other_indices = [pid - 1 for pid in other_ids]
    
    # Similarity weights: dot product between this person's affinity and all others
    weights = affinity_matrix[other_indices] @ person_vec
    
    # Add noise to weights so it's not purely deterministic
    weights += np.abs(np.random.normal(0, 0.01, len(weights)))
    weights = weights / weights.sum()  # normalize to probabilities
    
    connect_ids = np.random.choice(other_ids, size=number_of_connections,
                                   replace=False, p=weights)
    
    for conid in connect_ids:
        connective_lst.append({"FromID": person_id, "ToID": int(conid)})

connections = pd.DataFrame(connective_lst)
connections.head()

,FromID,ToID
0,1,52
1,1,240
2,2,146
3,2,163
4,2,246


In [9]:
#merging both people and pronouns into a singular excel sheet
affinity_cols = [c for c in people.columns if c.startswith("affinity_")]

merging = connections.merge(
    people, left_on="FromID", right_on="ID"
).merge(
    people, left_on="ToID", right_on="ID", suffixes=("_From", "_To")
)

final = merging[[
    "Name_From", "Pronouns_From", *[f"{c}_From" for c in affinity_cols],
    "Name_To", "Pronouns_To", *[f"{c}_To" for c in affinity_cols]
]]
final.head()

,Name_From,Pronouns_From,affinity_she_her_From,affinity_he_him_From,affinity_they_them_From,affinity_xe_xer_From,affinity_she_they_From,affinity_he_they_From,Name_To,Pronouns_To,affinity_she_her_To,affinity_he_him_To,affinity_they_them_To,affinity_xe_xer_To,affinity_she_they_To,affinity_he_they_To
0,Charles Gonzalez,xe/xer,0.406644,0.291097,0.099959,0.107525,0.090097,0.004678,Catherine Davis,she/her,0.607733,0.205325,0.021113,0.042351,0.046438,0.077041
1,Charles Gonzalez,xe/xer,0.406644,0.291097,0.099959,0.107525,0.090097,0.004678,Jessica Collins,she/they,0.388112,0.433052,0.103688,0.004747,0.054853,0.015547
2,Krista Delgado,he/they,0.334204,0.327193,0.207026,0.029623,0.055263,0.046691,Gregory Ross MD,he/they,0.127362,0.571650,0.140146,0.039596,0.025281,0.095964
3,Krista Delgado,he/they,0.334204,0.327193,0.207026,0.029623,0.055263,0.046691,Kristen Green,they/them,0.443682,0.306646,0.115257,0.068434,0.040529,0.025453
4,Krista Delgado,he/they,0.334204,0.327193,0.207026,0.029623,0.055263,0.046691,Angela Smith,she/her,0.300285,0.378663,0.198932,0.002725,0.109415,0.009979


In [10]:
# Map primary pronoun to user group label (for grouping/analysis)
def map_user_group(pronoun_str):
    pronoun_str = pronoun_str.lower()
    if pronoun_str in ["he/him", "he"]:
        return "he"
    elif pronoun_str in ["she/her", "she"]:
        return "she"
    elif pronoun_str in ["they/them", "they"]:
        return "they"
    elif pronoun_str in ["xe/xer", "xe"]:
        return "xe"
    elif pronoun_str in ["she/they"]:
        return "she/they"
    elif pronoun_str in ["he/they"]:
        return "he/they"
    else:
        return pronoun_str

people["User_Group"] = people["Pronouns"].apply(map_user_group)

# Explode into base pronoun rows for probability matrix
people["Base_Pronoun_List"] = people["User_Group"].apply(lambda x: x.split("/"))
exploded = people.explode("Base_Pronoun_List")

# Build probability matrix from sampled primary pronouns
pronoun_matrix = pd.crosstab(
    exploded["User_Group"],
    exploded["Base_Pronoun_List"],
    normalize="index"
)

pronoun_matrix = pronoun_matrix.reindex(
    columns=["he", "she", "they", "xe"], fill_value=0
)

group_counts = people["User_Group"].value_counts()
pronoun_matrix["User_Count"] = group_counts

cols = ["User_Count"] + [col for col in pronoun_matrix.columns if col != "User_Count"]
pronoun_matrix = pronoun_matrix[cols]

pronoun_matrix

Base_Pronoun_List,User_Count,he,she,they,xe
User_Group,,,,,
he,94,1.0,0.0,0.0,0.0
he/they,27,0.5,0.0,0.5,0.0
she,96,0.0,1.0,0.0,0.0
she/they,18,0.0,0.5,0.5,0.0
they,48,0.0,0.0,1.0,0.0
xe,17,0.0,0.0,0.0,1.0


In [11]:
# Summary stats on pronoun distribution
pronoun_counts = people["Pronouns"].value_counts()
pronoun_prob = people["Pronouns"].value_counts(normalize=True)
pronoun_percent = pronoun_prob * 100

print("Pronoun distribution (%):")
print(pronoun_percent.round(1))

# Connection pronoun probabilities
from_prob = final["Pronouns_From"].value_counts(normalize=True)
to_prob   = final["Pronouns_To"].value_counts(normalize=True)

# Average affinity similarity between connected pairs (sanity check for homophily)
from_affinities = merging[[f"{c}_From" for c in affinity_cols]].values
to_affinities   = merging[[f"{c}_To"   for c in affinity_cols]].values
dot_products = (from_affinities * to_affinities).sum(axis=1)
print(f"\nMean affinity similarity between connected pairs: {dot_products.mean():.4f}")
print(f"(Random baseline would be ~{(alpha/alpha.sum()**2).sum():.4f})")

Pronoun distribution (%):
Pronouns
she/her      32.0
he/him       31.3
they/them    16.0
he/they       9.0
she/they      6.0
xe/xer        5.7
Name: proportion, dtype: float64

Mean affinity similarity between connected pairs: 0.2544
(Random baseline would be ~0.0690)


In [12]:
#excel export
final.to_csv("people_connections_pronouns_dirichlet.csv", index=False, encoding='utf-8-sig')
pronoun_matrix.to_csv("pronoun_probability_group.csv")
people.to_csv("people_with_affinities.csv", index=False, encoding='utf-8-sig')

